<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 La Perilla de las Probabilidades 🎛️
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 06
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/06%20-%20Aprendizaje%20por%20Refuerzo/Para%20Dummies/02_Gradientes_de_Politica_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del [cuaderno 02 (Gradientes de política)](../02_Gradientes_de_Politica.ipynb). En el cuaderno anterior el agente aprendía **apuntando notas** y luego elegía la mejor. Aquí probamos **otra idea**: en lugar de llevar una libreta, el agente tiene una **perilla de probabilidades** («con qué frecuencia elijo cada opción») y **la va girando** según le vaya. Lo vamos a entender con una historia: **una máquina con tres botones**.

Al terminar podrás explicar:
1. Qué es una **política con probabilidades** y por qué a veces conviene **no ser predecible**.
2. La regla de aprendizaje: **si me fue bien, subo la probabilidad de lo que hice; si me fue mal, la bajo**.
3. Por qué es **difícil distinguir un premio bueno de simple buena suerte**, y cómo una **referencia** (compararte con tu promedio) ayuda.
4. En qué se diferencian **llevar una libreta**, **girar la perilla** y **hacer ambas cosas** (el «actor-crítico»).

> 🔗 Si luego quieres ver la versión con más detalle: [cuaderno estándar](../02_Gradientes_de_Politica.ipynb). Antes: [La libreta y el precipicio (Dummies)](01_Valor_y_QLearning_Dummies.ipynb).

---
## 1. La perilla: elegir «con cierta probabilidad» 🎲

Piensa en un **dado cargado**. En vez de decidir «siempre el botón 2», decides **con qué frecuencia** apretar cada botón: por ejemplo, 33% el botón 1, 33% el 2 y 33% el 3. Eso es una **política con probabilidades**, y la «perilla» es el conjunto de números que la define.

¿Por qué no simplemente elegir siempre el mejor botón? Por dos razones:
- **Al principio no sabes cuál es el mejor**; con una perilla repartida **pruebas de todo** (exploras) sin programarlo aparte.
- Hay situaciones en que **ser impredecible es lo óptimo**: en *piedra, papel o tijera*, quien siempre juega lo mismo pierde; la mejor estrategia es **repartir al azar**. Una libreta con «la mejor opción» no sabe hacer eso; **una perilla sí**.

**La regla para girar la perilla** es de sentido común:

> 🔼 Si apreté un botón y **me fue mejor de lo que esperaba**, **subo la probabilidad** de apretarlo.
> 🔽 Si me fue **peor de lo que esperaba**, **la bajo**.

Probemos con una máquina de **3 botones**. Cada botón da, en promedio, **10, 11 y 10.5 pesos**, pero cada vez que lo aprietas el premio **varía mucho por pura suerte** (a veces 14, a veces 7). El botón 2 es el mejor, pero la diferencia es pequeña frente al ruido. Dejamos que el agente juegue **1 000 veces** girando su perilla, solo con **un intento** para ver su evolución.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

MEDIAS, RUIDO = np.array([10.0, 11.0, 10.5]), 2.0          # premio medio de cada boton y cuanto varia por suerte
BOTON_BUENO = int(np.argmax(MEDIAS))                        # el boton 2 (indice 1)

def probabilidades(perilla):
    e = np.exp(perilla - perilla.max()); return e / e.sum()

def jugar(pasos=1000, paso_perilla=0.05, referencia=False, semilla=0):
    """Gira la perilla pasos veces. referencia=True: compara cada premio con el promedio de premios recientes."""
    rng = np.random.default_rng(semilla)
    perilla, promedio, historia = np.zeros(3), 0.0, []
    for _ in range(pasos):
        p = probabilidades(perilla)
        boton = rng.choice(3, p=p); premio = rng.normal(MEDIAS[boton], RUIDO)
        mejor_que_lo_esperado = premio - (promedio if referencia else 0.0)      # sin referencia: cualquier premio positivo "se siente bien"
        empuje = -p * mejor_que_lo_esperado; empuje[boton] += mejor_que_lo_esperado   # sube lo que apreto (si fue bueno) y baja lo demas
        perilla += paso_perilla * empuje
        promedio += 0.1 * (premio - promedio)
        historia.append(p)
    return np.array(historia)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.4), sharey=True)
for ax, ref, titulo in [(axes[0], False, "Sin referencia (un intento)"), (axes[1], True, "Con referencia: compararse con el promedio (un intento)")]:
    h = jugar(referencia=ref, semilla=3)
    for b, color in zip(range(3), ["#dc2626", "#16a34a", "#2563eb"]):
        ax.plot(h[:, b], color=color, lw=2.2, label=f"botón {b + 1}" + (" (el mejor)" if b == BOTON_BUENO else ""))
    ax.set_xlabel("Veces que se juega"); ax.set_title(titulo, fontweight="bold", fontsize=10); ax.legend(fontsize=9)
axes[0].set_ylabel("Probabilidad de apretar cada botón")
plt.tight_layout(); plt.show()
print("En ESTE intento hasta el agente sin referencia tuvo suerte y se quedó con el botón bueno: un solo intento puede engañar. Para saber qué ocurre DE VERDAD hay que repetir el experimento muchas veces (siguiente sección).")

---
## 2. ¿Premio bueno o simple suerte? La referencia 🎓

Saca una **nota de 4.5** en un examen. ¿Es buena? **Depende del curso**: si el promedio fue 2.5, ¡excelente!; si el promedio fue 4.6, estuviste **por debajo**. Una nota **sola** no dice si lo hiciste bien; **comparada con lo esperado** sí.

En nuestra máquina pasa lo mismo: **todos los premios son positivos** (alrededor de 10). Sin una referencia, **cualquier botón que aprietes «se siente bien»** (¡ganaste 9 pesos!) y su probabilidad **sube**, aunque fuera de los peores. Entonces el agente puede **encariñarse con el primer botón que tuvo suerte**, sea o no el mejor. Con una **referencia** (el promedio de lo que ha ganado hasta ahora), solo sube lo que fue **mejor que el promedio**, y baja lo que fue peor.

Para no confundirnos con la suerte de un solo intento, repetimos todo **40 veces** (40 agentes nuevos, sin y con referencia) y miramos **cuántos terminan prefiriendo con claridad el botón bueno** (probabilidad mayor a 80%) y cuántos se **quedan «enamorados» de un botón malo** (menos del 20% para el bueno).

In [ ]:
N_AGENTES, PASOS = 40, 1000
finales = {}
curvas = {}
for ref in (False, True):
    H = np.array([jugar(PASOS, referencia=ref, semilla=100 + k) for k in range(N_AGENTES)])      # (agentes, pasos, 3)
    curvas[ref] = H[:, :, BOTON_BUENO]; finales[ref] = H[:, -1, BOTON_BUENO]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))
for ref, color, nombre in [(False, "#dc2626", "Sin referencia"), (True, "#16a34a", "Con referencia (promedio)")]:
    m, sd = curvas[ref].mean(axis=0), curvas[ref].std(axis=0)
    axes[0].plot(m, color=color, lw=2.5, label=nombre); axes[0].fill_between(np.arange(PASOS), np.clip(m - sd, 0, 1), np.clip(m + sd, 0, 1), color=color, alpha=0.15)
axes[0].set_xlabel("Veces que se juega"); axes[0].set_ylabel("Probabilidad de apretar el botón bueno (media ± desv. de 40 agentes)")
axes[0].set_title("El botón bueno se aprende mucho mejor con referencia", fontweight="bold", fontsize=10); axes[0].legend()
etiquetas = ["Sin referencia", "Con referencia"]
claro = [100 * np.mean(finales[r] > 0.8) for r in (False, True)]; mal = [100 * np.mean(finales[r] < 0.2) for r in (False, True)]
x = np.arange(2); axes[1].bar(x - 0.18, claro, 0.36, color="#16a34a", label="Prefiere claramente el botón BUENO")
axes[1].bar(x + 0.18, mal, 0.36, color="#dc2626", label="Se quedó enamorado de un botón MALO")
axes[1].set_xticks(x); axes[1].set_xticklabels(etiquetas); axes[1].set_ylabel("% de los 40 agentes"); axes[1].legend(fontsize=9)
axes[1].set_title("Al final del entrenamiento, ¿qué prefiere cada agente?", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()

print(f"Sin referencia: {claro[0]:.0f}% prefiere el botón bueno y {mal[0]:.0f}% se quedó con uno malo.")
print(f"Con referencia: {claro[1]:.0f}% prefiere el botón bueno y {mal[1]:.0f}% se quedó con uno malo.")
assert claro[1] > claro[0] + 20 and mal[1] < mal[0]                    # la referencia ayuda de forma clara

---
## 3. Tres maneras de aprender: libreta, perilla o ambas 📓🎛️

| | 📓 **La libreta** (Q-learning) | 🎛️ **La perilla** (gradiente de política) | 🤝 **Ambas** (actor-crítico) |
|---|---|---|---|
| **Qué lleva el agente** | Una nota por situación y movimiento | Probabilidades de elegir cada opción | Perilla **y** libreta |
| **Cómo decide** | Elige la opción de **mejor nota** | **Tira el dado** con sus probabilidades | Tira el dado; la libreta le dice si le fue **mejor o peor de lo esperado** |
| **Ventaja** | Aprende rápido con pocos datos | Sirve cuando las opciones son **infinitas** (¿cuánto giro el volante?) y cuando **conviene ser impredecible** | Suma lo bueno de ambas: **menos ruido** que la perilla sola |
| **Desventaja** | Mal con opciones infinitas; siempre decide lo mismo | **Muy ruidoso** (¡como viste!) y lento | Hay que entrenar **dos** cosas |
| **Dónde se usa** | Videojuegos de Atari | Robótica, control suave | Casi todo lo moderno (robots, ajustar chatbots con opiniones humanas) |

La **referencia** de la sección anterior es el puente entre ambas: compararte con lo que **esperabas** te lo dice la libreta. Por eso en el cuaderno estándar la llamamos *ventaja*: **cuánto mejor (o peor) resultó una acción que el promedio de la situación**.

> ⚠️ **Honestidad:** la perilla es **ruidosa y depende de la suerte**: sin referencia, una gran parte de los agentes se quedó «enamorada» de un botón malo, y aunque con referencia casi todos acertaron **en este experimento**, con menos jugadas o más ruido podría no ser así. Por eso se repiten los experimentos muchas veces antes de creer en un resultado.

---
##### 🛠️ Práctica 1: Gira la perilla a mano

Giremos la perilla de la máquina de 3 botones **un solo paso**, con números, usando la función `probabilidades` de arriba. La perilla empieza en cero y el paso de giro es `0.05`.

1. Crea `perilla = np.zeros(3)` y calcula `p = probabilidades(perilla)`: debe repartir 1/3 a cada botón.
2. Aprietas el **botón 2** (posición `1`) y ganas **14 pesos**; tu promedio es **10**. Calcula `mejor = 14 - 10`. El empuje es `-p * mejor` en todos los botones y luego **sumas `mejor` al botón apretado** (así sube el que apretaste y bajan los demás). Gira: `perilla_buena = perilla + 0.05 * empuje` y guarda `p_despues_bueno = probabilidades(perilla_buena)`.
3. Repite desde cero (perilla en cero) con un premio de **7** (peor que tu promedio) y guarda `p_despues_malo`.
4. Comprueba que tras el premio bueno **subió** la probabilidad del botón 2 y bajaron las otras dos; y que tras el premio malo **bajó** la del botón 2.

In [ ]:
# Escribe tu código aquí

perilla = np.zeros(3)
p = probabilidades(perilla)                      # debe ser 1/3, 1/3, 1/3

# Premio BUENO: aprietas el boton 2 (posicion 1), ganas 14 y tu promedio es 10
# mejor = 14 - 10
# empuje = -p * mejor                            # baja todos...
# empuje[1] += mejor                             # ...y sube el boton que apretaste
# perilla_buena = perilla + 0.05 * empuje
# p_despues_bueno = probabilidades(perilla_buena)

# Premio MALO: lo mismo, pero con un premio de 7
# mejor = 7 - 10
# ...
# p_despues_malo = ...

# Comprobaciones con assert


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
perilla = np.zeros(3)
p = probabilidades(perilla)                      # debe ser 1/3, 1/3, 1/3

# Premio BUENO: aprietas el boton 2 (posicion 1), ganas 14 y tu promedio es 10
mejor = 14 - 10
empuje = -p * mejor                              # baja todos...
empuje[1] += mejor                               # ...y sube el boton que apretaste
perilla_buena = perilla + 0.05 * empuje
p_despues_bueno = probabilidades(perilla_buena)

# Premio MALO: lo mismo, pero con un premio de 7
mejor = 7 - 10
empuje = -p * mejor
empuje[1] += mejor
perilla_mala = perilla + 0.05 * empuje
p_despues_malo = probabilidades(perilla_mala)

print("Antes:                ", p.round(3))
print("Despues del premio 14:", p_despues_bueno.round(3))
print("Despues del premio 7: ", p_despues_malo.round(3))

# Comprobaciones
assert np.allclose(p, 1 / 3)
assert p_despues_bueno[1] > p[1] and (p_despues_bueno[[0, 2]] < p[[0, 2]]).all()
assert p_despues_malo[1] < p[1]
print("Mejor de lo esperado: sube la probabilidad del boton apretado. Peor de lo esperado: baja.")
```
</details>

---


---
## Resumen en una frase 🎯

> **En los *gradientes de política* el agente tiene una *perilla de probabilidades* y la gira: sube la probabilidad de lo que le fue mejor de lo esperado y baja la de lo que le fue peor; como el azar confunde, compararse con una *referencia* (el promedio) reduce el ruido, y combinar perilla con libreta da los métodos *actor-crítico* que hoy se usan en casi todo.**

### Lo que aprendiste hoy:
- ✅ Una **política con probabilidades** explora sola y puede ser **impredecible a propósito** (piedra, papel, tijera).
- ✅ Regla de la perilla: **sube lo que salió mejor de lo esperado, baja lo peor**.
- ✅ Una nota sola no basta: **compárala con lo que esperabas** (la referencia).
- ✅ Sin referencia, el agente puede **enamorarse de un botón malo** por simple suerte.
- ✅ **Libreta + perilla** = *actor-crítico*, la base de los métodos modernos.

> 🎓 **Siguiente paso:** [El cerebro, las fichas de estudio y el profesor paciente (Dummies)](03_Deep_RL_Dummies.ipynb). Versión con más detalle de este tema: [cuaderno estándar](../02_Gradientes_de_Politica.ipynb).

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿por qué una nota de 4.5 «sola» no dice si fue un buen examen? ¿Qué papel juega la *referencia* en la máquina de los tres botones?
2. Piensa en un caso de tu vida o tu trabajo donde convenga **ser impredecible** (una estrategia de negociación, un juego, la ruta de un repartidor). ¿Por qué una «libreta» que siempre elige la mejor opción lo haría mal?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>